# Intelligent Geospatial Sampling Quality Control — Powered by XGBlassifier
by Muhammad Erico Ricardo

# 1. Import Libraries

In [1]:
pip install -U imbalanced-learn scikit-learn


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# 1. Standard Library
import pickle

# 2. Data Manipulation & Numerical Operations
import pandas as pd
import numpy as np
from imblearn.over_sampling import SMOTE

# 3. Visualization & Correlation Analysis
import matplotlib.pyplot as plt
import seaborn as sns
import phik
from phik.report import plot_correlation_matrix

# 4. Geospatial Data Handling
import geopandas as gpd
from shapely import wkt
from shapely.geometry import Point

# 5. Machine Learning - Preprocessing & Model Selection
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# 6. Machine Learning - Models
from xgboost import XGBClassifier
from lazypredict.Supervised import LazyClassifier

# 7. Machine Learning - Metrics
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    roc_auc_score
)

# 8. Hyperparameter Tuning
from sklearn.model_selection import GridSearchCV

# 2. Data Loading and Understanding

## 2.1 Load the data

Load data from manual validation process 

In [3]:
# Load data from CSV
Data = pd.read_csv(r"C:\Users\Rico\Downloads\Modeling Project\processed_polygon_validation_data_New_V4.csv")

In [4]:
# See the columns in the dataset
Data.columns

Index(['Kegiatan', 'TRANSNO', 'OCCODE', 'BLOCKCODE', 'SUBBLOCKCODE', 'AKURASI',
       'JUMLAH_SATELIT', 'IS_VALID_SAMPLING', 'SCANON', 'VALIDATION_NOTE',
       'ISCHECK', 'SHAPE', 'Latitude_1', 'Longitude_1', 'Latitude_2',
       'Longitude_2', 'Latitude_3', 'Longitude_3', 'geometry', 'in_1', 'in_2',
       'in_3', 'all_inside', 'dist_c1', 'dist_c2', 'dist_c3', 'angle_1',
       'angle_2', 'angle_3', 'movement_angle_12', 'movement_angle_23',
       'validation_x', 'validation_y'],
      dtype='object')

In [5]:
# See the data types of each column
Data.dtypes

Kegiatan              object
TRANSNO               object
OCCODE                object
BLOCKCODE             object
SUBBLOCKCODE          object
AKURASI              float64
JUMLAH_SATELIT       float64
IS_VALID_SAMPLING      int64
SCANON                object
VALIDATION_NOTE       object
ISCHECK                int64
SHAPE                 object
Latitude_1           float64
Longitude_1          float64
Latitude_2           float64
Longitude_2          float64
Latitude_3           float64
Longitude_3          float64
geometry              object
in_1                   int64
in_2                   int64
in_3                   int64
all_inside             int64
dist_c1              float64
dist_c2              float64
dist_c3              float64
angle_1              float64
angle_2              float64
angle_3              float64
movement_angle_12    float64
movement_angle_23    float64
validation_x          object
validation_y          object
dtype: object

In [6]:
Data.head()

,Kegiatan,TRANSNO,OCCODE,BLOCKCODE,SUBBLOCKCODE,AKURASI,JUMLAH_SATELIT,IS_VALID_SAMPLING,SCANON,VALIDATION_NOTE,...,dist_c1,dist_c2,dist_c3,angle_1,angle_2,angle_3,movement_angle_12,movement_angle_23,validation_x,validation_y
0,Harrowing,Q08GF-GQ-011025-001,G,TU4/17,164TU032,3.00,32.00,1,2025-10-01 10:18:47.000,"Status: Valid, Sudut sample 1 ke 2: 57.08, Sud...",...,102.22,29.80,53.84,-136.81,-5.98,12.83,32.70,33.37,y,y
1,Harrowing,Q08GF-GQ-021025-002,G,TU3/17,162TU028,3.00,33.00,1,2025-10-02 10:48:11.000,"Status: Valid, Sudut sample 1 ke 2: 147.7, Sud...",...,84.86,55.21,77.92,128.86,-67.48,-61.48,-57.57,-47.38,y,y
2,Harrowing,Q08GF-GQ-021025-001,G,TU3/17,162TU029,3.00,34.00,1,2025-10-02 10:39:20.000,"Status: Valid, Sudut sample 1 ke 2: 214.53, Su...",...,76.37,37.38,61.77,70.21,-156.59,-153.90,-124.75,-149.81,y,y
3,Harrowing,Q08GF-GQ-011025-002,G,TU3/17,162TU024,3.00,25.00,1,2025-10-01 13:24:43.000,"Status: Valid, Sudut sample 1 ke 2: 123.93, Su...",...,41.27,7.63,28.37,156.72,-115.05,-32.80,-33.80,-17.34,y,y
4,Keremahan,Q14GF-GQ-151125-001,G,TU2/16,152TU011,3.00,27.00,1,2025-11-15 08:42:44.000,"Status: Valid, Sudut sample 1 ke 2: 314.82, Su...",...,36.91,69.56,92.73,-126.81,167.04,160.19,135.32,140.87,y,y


In [7]:
# Rename validation_y to validation
Data.rename(columns={'validation_y': 'validation'}, inplace=True)

In [8]:
Data.head()

,Kegiatan,TRANSNO,OCCODE,BLOCKCODE,SUBBLOCKCODE,AKURASI,JUMLAH_SATELIT,IS_VALID_SAMPLING,SCANON,VALIDATION_NOTE,...,dist_c1,dist_c2,dist_c3,angle_1,angle_2,angle_3,movement_angle_12,movement_angle_23,validation_x,validation
0,Harrowing,Q08GF-GQ-011025-001,G,TU4/17,164TU032,3.00,32.00,1,2025-10-01 10:18:47.000,"Status: Valid, Sudut sample 1 ke 2: 57.08, Sud...",...,102.22,29.80,53.84,-136.81,-5.98,12.83,32.70,33.37,y,y
1,Harrowing,Q08GF-GQ-021025-002,G,TU3/17,162TU028,3.00,33.00,1,2025-10-02 10:48:11.000,"Status: Valid, Sudut sample 1 ke 2: 147.7, Sud...",...,84.86,55.21,77.92,128.86,-67.48,-61.48,-57.57,-47.38,y,y
2,Harrowing,Q08GF-GQ-021025-001,G,TU3/17,162TU029,3.00,34.00,1,2025-10-02 10:39:20.000,"Status: Valid, Sudut sample 1 ke 2: 214.53, Su...",...,76.37,37.38,61.77,70.21,-156.59,-153.90,-124.75,-149.81,y,y
3,Harrowing,Q08GF-GQ-011025-002,G,TU3/17,162TU024,3.00,25.00,1,2025-10-01 13:24:43.000,"Status: Valid, Sudut sample 1 ke 2: 123.93, Su...",...,41.27,7.63,28.37,156.72,-115.05,-32.80,-33.80,-17.34,y,y
4,Keremahan,Q14GF-GQ-151125-001,G,TU2/16,152TU011,3.00,27.00,1,2025-11-15 08:42:44.000,"Status: Valid, Sudut sample 1 ke 2: 314.82, Su...",...,36.91,69.56,92.73,-126.81,167.04,160.19,135.32,140.87,y,y


## 2.2 Choose the relevant columns for modeling

In [9]:
# Choose the relevant columns for modeling
gdf = Data.loc[:, ['TRANSNO', 'geometry', 'validation', 'AKURASI', 'Kegiatan','JUMLAH_SATELIT','SHAPE','Latitude_1','Longitude_1','Latitude_2','Longitude_2','Latitude_3','Longitude_3','in_1','in_2','in_3','dist_c1','dist_c2','dist_c3', 'angle_1','angle_2','angle_3','movement_angle_12', 'movement_angle_23',
       'validation']]

In [10]:
# Change the validation column to binary (0 and 1)
gdf['validation'] = Data['validation'].apply(lambda x: 1 if x == 'y' else 0)

In [11]:
gdf.columns

Index(['TRANSNO', 'geometry', 'validation', 'AKURASI', 'Kegiatan',
       'JUMLAH_SATELIT', 'SHAPE', 'Latitude_1', 'Longitude_1', 'Latitude_2',
       'Longitude_2', 'Latitude_3', 'Longitude_3', 'in_1', 'in_2', 'in_3',
       'dist_c1', 'dist_c2', 'dist_c3', 'angle_1', 'angle_2', 'angle_3',
       'movement_angle_12', 'movement_angle_23', 'validation'],
      dtype='object')

In [12]:
# Save the cleaned and processed data to a new CSV file
gdf.to_csv("check_data.csv", index=False)

# 3. Pick Best Model Using Lazy Predict

## 3.1 Load Spatial and Feature Engineering

In [13]:
df = pd.read_csv('check_data.csv')

# Konversi WKT dan Proyeksi ke Meter (UTM 48S) untuk akurasi kalkulasi
df['geometry'] = df['SHAPE'].apply(wkt.loads)
gdf = gpd.GeoDataFrame(df, geometry='geometry', crs="EPSG:4326")
gdf_meter = gdf.to_crs(epsg=32748)
centroids_meter = gdf_meter.geometry.centroid
# Ubah AKURASI menjadi 'GPS_PRECISION_SCORE' dengan skala 0-100 (semakin kecil AKURASI, semakin tinggi skor)
# gdf['GPS_PRECISION_SCORE'] = (1 - (gdf['AKURASI'] / gdf['AKURASI'].max())) * 100
gdf['GPS_PRECISION_SCORE'] = np.clip(100 - (gdf['AKURASI'] * 10), 0, 100)


def calculate_angle(p1, p2):
    return np.degrees(np.arctan2(p2.y - p1.y, p2.x - p1.x))

# Iterasi untuk Point 1, 2, dan 3
for i in range(1, 4):
    lat, lon = f'Latitude_{i}', f'Longitude_{i}'
    # Buat geometri titik
    p_geom = [Point(xy) for xy in zip(gdf[lon], gdf[lat])]
    p_gdf = gpd.GeoDataFrame(geometry=p_geom, crs="EPSG:4326", index=gdf.index).to_crs(epsg=32748)
    
    # Fitur: Is Inside, Distance, dan Angle ke Centroid
    gdf[f'in_{i}'] = p_gdf.within(gdf_meter.geometry).astype(int)
    gdf[f'dist_c{i}'] = p_gdf.distance(centroids_meter)
    gdf[f'angle_{i}'] = [calculate_angle(c, p) for c, p in zip(centroids_meter, p_gdf.geometry)]
    
    # Simpan sementara untuk kalkulasi movement angle
    if i == 1: p1_meter = p_gdf.geometry
    if i == 2: p2_meter = p_gdf.geometry
    if i == 3: p3_meter = p_gdf.geometry

# Fitur Tambahan: All Inside & Movement Angles (Diagonalitas)
gdf['all_inside'] = ((gdf['in_1'] == 1) & (gdf['in_2'] == 1) & (gdf['in_3'] == 1)).astype(int)
gdf['move_angle_12'] = [calculate_angle(p1, p2) for p1, p2 in zip(p1_meter, p2_meter)]
gdf['move_angle_23'] = [calculate_angle(p2, p3) for p2, p3 in zip(p2_meter, p3_meter)]

### 1. Transforming the World into Meters

The journey begins by loading the data and converting the `SHAPE` text field (WKT format) into a tangible geometric object. However, degree coordinates (WGS84 - EPSG:4326) are not accurate for calculating distances.

* **Action:** We project the data into **UTM Zone 48S (EPSG:32748)**.
* **Goal:** To convert the units from degrees to **meters**, making distance and area calculations precise and physically relevant.
* **Centroid:** We determine the center point (*centroid*) of each polygon as the primary reference point.

### 2. Iterating Over Three Primary Points

Our data contains three coordinate pairs (Point 1, 2, and 3). Through a *loop*, we process each of them one by one in a consistent manner:

1. **Geolocating:** Converts the Latitude/Longitude pairs into `Point` objects.

2. **Reprojection:** Aligns the coordinate system of the points to meters to align with our area data.
3. **Spatial Relationship:**
* **Is Inside? (`in_i`):** Checks whether the point is inside or outside the polygon (Binary: 0 or 1).
* **Distance (`dist_ci`):** Calculates the distance of the point from the center of the polygon in meters.
* **Relative Angle (`angle_i`):** Calculates the directional angle from the center of the polygon to the point using the trigonometric function `arctan2`.

### 3. Collective Analysis and Movement Dynamics

Once the characteristics of each point are obtained, we see the big picture through the combined features:

* **The "All Inside" Status:** We create the `all_inside` feature. If all three points are inside the polygon area, then the line is considered to have perfect spatial integrity.
* **Movement Angles (Vector):** This is the most interesting part. We calculate the angle of movement from **Point 1 to Point 2**, and then from **Point 2 to Point 3**.
* **Insight:** The `move_angle_12` and `move_angle_23` features help the model understand the "direction of travel" or "diagonality" of the point sequence. This is very useful for detecting anomalous movement patterns or specific directional trends in the data.

## 3.2 Select Numeric Feature Only

In [14]:
features = [
    'in_1', 'in_2', 'in_3', 'all_inside',
    'dist_c1', 'dist_c2', 'dist_c3',
    'angle_1', 'angle_2', 'angle_3',
    'move_angle_12', 'move_angle_23',
    'GPS_PRECISION_SCORE', 'JUMLAH_SATELIT'
]

X = gdf[features]
y = gdf['validation']  # Target: 1 untuk valid, 0 untuk tidak valid

Since we will run LazyPredict first to shortlist a model, every feature used must be numeric.

## 3.3 Pipeline Preprocessing

In [15]:
# Mengatasi Missing Values dengan Median dan melakukan Scaling
imputer = SimpleImputer(strategy='median')
scaler = StandardScaler()

X_cleaned = imputer.fit_transform(X)
X_scaled = scaler.fit_transform(X_cleaned)

## 3.4 Split The Data

In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

In [17]:
from imblearn.over_sampling import SMOTE

# Inisialisasi SMOTE
smote = SMOTE(random_state=42)

# Langsung masukkan X_train dan y_train yang terpisah
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)


In [18]:
import pandas as pd

# Cek sebelum SMOTE
print("Sebelum SMOTE:")
print(pd.Series(y_train).value_counts())

# Cek setelah SMOTE
print("\nSesudah SMOTE:")
print(pd.Series(y_train_resampled).value_counts())


Sebelum SMOTE:
validation
1    2156
0     973
Name: count, dtype: int64

Sesudah SMOTE:
validation
1    2156
0    2156
Name: count, dtype: int64


## 3.5 Training Lazy Predict Model

In [19]:
clf = LazyClassifier(verbose=0, ignore_warnings=True, predictions=False)
models, _ = clf.fit(X_train_resampled, X_test, y_train_resampled, y_test)

# Tampilkan hasil berdasarkan ROC AUC (karena data imbalanced)
print("\n--- Model Performance Results ---")
display(models.sort_values(by='ROC AUC', ascending=False))

  0%|          | 0/32 [00:00<?, ?it/s]

[LightGBM] [Info] Number of positive: 2156, number of negative: 2156
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000860 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2566
[LightGBM] [Info] Number of data points in the train set: 4312, number of used features: 14
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000

--- Model Performance Results ---


,Accuracy,Balanced Accuracy,ROC AUC,F1 Score,Time Taken
Model,,,,,
RandomForestClassifier,0.96,0.95,0.95,0.96,0.41
XGBClassifier,0.96,0.95,0.95,0.96,0.15
LGBMClassifier,0.96,0.95,0.95,0.96,0.13
ExtraTreesClassifier,0.96,0.94,0.94,0.95,0.20
BaggingClassifier,0.94,0.93,0.93,0.94,0.14
DecisionTreeClassifier,0.93,0.93,0.93,0.94,0.03
KNeighborsClassifier,0.94,0.93,0.93,0.94,0.04
LabelPropagation,0.94,0.92,0.92,0.94,0.45
LabelSpreading,0.94,0.92,0.92,0.94,0.57


## 3.6 Model Evaluation
Based from Lazy Predict, the best classification model is XGBClassifier because it has same performance with LGBMClassifier but with less time taken. So we going to use XGBClassifier to continue this project.

# 4. Extreme Gradient Boosting Classifier Model

## 4.1 Model Explanation 

XGBClassifier (Extreme Gradient Boosting) is an ensemble algorithm that falls into the Boosting category. As the name suggests, it is a highly optimized and "extreme" implementation of Gradient Boosting Trees designed for speed and performance.

This model works by building multiple decision trees sequentially (one after another), where each new tree is specifically trained to correct the errors made by the previous trees to determine the final prediction.

### How It Works:

1. **Sequential Learning:** Unlike Bagging algorithms that build trees independently, XGBoost builds trees in a chain. It starts with a base prediction, calculates the residuals (errors), and trains the next tree to predict those residuals.
2. **Gradient Descent Optimization:** It uses a gradient descent algorithm to minimize the loss function (such as LogLoss for classification) when adding new trees, ensuring each step moves closer to the optimal solution.
3. **Regularization (L1 & L2):** To prevent overfitting, XGBoost includes built-in regularization ($\text{L1 / Lasso}$ and $\text{L2 / Ridge}$). This penalizes complex trees, making it much more robust against noise compared to standard Gradient Boosting.

---

## Comparison: XGBoost vs. Random Forest vs. Extra Trees

To understand the differences, we need to look at how each model handles data.

| Features | XGBoost (XGB) | Random Forest (RF) | Extra Trees (ET) |
| --- | --- | --- | --- |
| **Category** | Boosting (Gradient Boosting) | Bagging | Bagging (Extremely Randomized) |
| **How it Works** | Builds trees sequentially (new trees correct errors in previous trees). | Builds trees in parallel and independently. | Similar to RF, but more random in determining splits. |
| **Split Determination** | Finds splits based on decreasing loss function (gradient). | Finds the mathematically optimal split. | Chooses splits randomly. |
| **Variance & Bias** | Focuses on reducing bias first, then controls variance via regularization. | Focuses on reducing variance (overfitting). | Reduces variance more drastically than RF. |
| **Speed** | Moderate to Fast (supports parallel processing on CPU/GPU, but inherently sequential). | Fairly fast. | **The fastest** (because it doesn't need to calculate the optimal split). |


## 4.2 Split Data to Train, Test and Val Data

In [20]:
# Val & Test diambil dari X_test/y_test ASLI (belum pernah disentuh SMOTE)
# Train (resampled) TIDAK diapa-apakan lagi — sudah final dari langkah SMOTE sebelumnya
X_val, X_test, y_val, y_test = train_test_split(
    X_test, y_test,          # <-- sumbernya data test ASLI, bukan X_train_resampled
    test_size=0.50,
    random_state=42,
    stratify=y_test
)

print("Ukuran data:")
print(f"Train (resampled) : {X_train_resampled.shape[0]}")
print(f"Val   (asli)      : {X_val.shape[0]}")
print(f"Test  (asli)      : {X_test.shape[0]}\n")

Ukuran data:
Train (resampled) : 4312
Val   (asli)      : 391
Test  (asli)      : 392



In [21]:
# X_train_resampled, X_temp, y_train_resampled, y_temp = train_test_split(
#     X_train_resampled, y_train_resampled,
#     test_size=0.30,
#     random_state=42,
#     stratify=y_train_resampled
# )

# X_val, X_test, y_val, y_test = train_test_split(
#     X_temp, y_temp,
#     test_size=0.50,
#     random_state=42,
#     stratify=y_temp
# )

# print("Ukuran data:")
# print(f"Train : {X_train_resampled.shape[0]}")
# print(f"Val   : {X_val.shape[0]}")
# print(f"Test  : {X_test.shape[0]}\n")

## 4.3 Create a Pipeline

In [22]:
pipeline_et = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('classifier', XGBClassifier(
        n_estimators=100,
        random_state=42
    ))
])

## 4.4 Train The Model

In [23]:
pipeline_et.fit(X_train_resampled, y_train_resampled)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('scaler', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,14
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in t

In [24]:
# pipeline_et.fit(X_train, y_train)

## 4.5 Create Evaluation Function

In [25]:
def evaluate_model(model, X_eval, y_eval, name="SET"):
    y_pred = model.predict(X_eval)
    
    # probabilitas untuk ROC-AUC
    y_prob = None
    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_eval)[:, 1]

    cm = confusion_matrix(y_eval, y_pred)

    print(f"\n================ {name} ================")
    print("Confusion Matrix:")
    print(cm)

    print("\nClassification Report:")
    print(classification_report(y_eval, y_pred, digits=4))

    results = {
        "accuracy": accuracy_score(y_eval, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_eval, y_pred),
        "precision": precision_score(y_eval, y_pred, zero_division=0),
        "recall": recall_score(y_eval, y_pred, zero_division=0),
        "f1_score": f1_score(y_eval, y_pred, zero_division=0),
    }

    if y_prob is not None:
        results["roc_auc"] = roc_auc_score(y_eval, y_prob)

    print("Ringkasan Metrik:")
    for k, v in results.items():
        print(f"{k:>18}: {v:.4f}")

    return results, cm, y_pred

## 4.6 Evaluation at Validation Set

In [26]:
val_results, val_cm, val_pred = evaluate_model(pipeline_et, X_val, y_val, "VALIDATION")


================ VALIDATION ================
Confusion Matrix:
[[111  10]
 [  7 263]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9407    0.9174    0.9289       121
           1     0.9634    0.9741    0.9687       270

    accuracy                         0.9565       391
   macro avg     0.9520    0.9457    0.9488       391
weighted avg     0.9563    0.9565    0.9564       391

Ringkasan Metrik:
          accuracy: 0.9565
 balanced_accuracy: 0.9457
         precision: 0.9634
            recall: 0.9741
          f1_score: 0.9687
           roc_auc: 0.9805


## 4.7 Evaluation at Test Set

In [27]:
test_results, test_cm, test_pred = evaluate_model(pipeline_et, X_test, y_test, "TEST")


================ TEST ================
Confusion Matrix:
[[114   8]
 [  4 266]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9661    0.9344    0.9500       122
           1     0.9708    0.9852    0.9779       270

    accuracy                         0.9694       392
   macro avg     0.9685    0.9598    0.9640       392
weighted avg     0.9693    0.9694    0.9692       392

Ringkasan Metrik:
          accuracy: 0.9694
 balanced_accuracy: 0.9598
         precision: 0.9708
            recall: 0.9852
          f1_score: 0.9779
           roc_auc: 0.9794


## 4.8 Save Model Before Hyperparameter Tuning

In [28]:
model_filename = 'xgboost_spatial_model_New.pkl'
with open(model_filename, 'wb') as file:
    pickle.dump(pipeline_et, file)

print(f"\nModel XGBoost berhasil diekspor ke: {model_filename}")


Model XGBoost berhasil diekspor ke: xgboost_spatial_model_New.pkl


# 5. Hyperparameter Tuning

## 5.1 Define The Parameter

In [29]:
param_grid = {
    # 1. Parameter Utama (Memasukkan Nilai Default)
    'classifier__n_estimators': [100, 200],           # Default: 100
    'classifier__max_depth': [4, 6, 7],               # Default: 6 (Sebelumnya tidak ada nilai 6)
    'classifier__learning_rate': [0.1, 0.2, 0.3],     # Default: 0.3 (Sebelumnya tidak ada nilai 0.3)
    'classifier__subsample': [0.8, 1.0],              # Default: 1.0
    'classifier__colsample_bytree': [0.8, 1.0],       # Default: 1.0
    
    # 2. Pengayaan untuk Kontrol Overfitting (Regularisasi)
    'classifier__min_child_weight': [1, 3, 5],        # Default: 1 (Nilai lebih tinggi = lebih konservatif)
    'classifier__gamma': [0, 0.1, 0.2]                # Default: 0 (Nilai lebih tinggi = mencegah overfitting)
}

## 5.2 GridSearchCV Intiation

In [30]:
grid_search = GridSearchCV(
    estimator=pipeline_et,       # Masukkan pipeline utuh
    param_grid=param_grid,       # Masukkan kamus parameter
    cv=5,                        # 5-Fold Cross Validation (membagi data latih jadi 5 bagian)
    scoring='f1_macro',          # Fokus pada F1 Score rata-rata untuk semua kelas
    n_jobs=-1,                   # Gunakan SELURUH core prosesor komputer untuk komputasi paralel
    verbose=2                    # Menampilkan progres log saat berjalan
)

## 5.3 Execute The tuning

In [31]:
from joblib import parallel_backend

print("Memulai proses hyperparameter tuning dengan threading backend...")

# Membungkus proses fit agar aman di Windows + Python 3.13
with parallel_backend('threading', n_jobs=-1):
    grid_search.fit(X_train_resampled, y_train_resampled)

print("Selesai!")
print("Best Params:", grid_search.best_params_)

Memulai proses hyperparameter tuning dengan threading backend...
Fitting 5 folds for each of 648 candidates, totalling 3240 fits
[CV] END classifier__colsample_bytree=0.8, classifier__gamma=0, classifier__learning_rate=0.1, classifier__max_depth=4, classifier__min_child_weight=1, classifier__n_estimators=100, classifier__subsample=1.0; total time=   0.7s
[CV] END classifier__colsample_bytree=0.8, classifier__gamma=0, classifier__learning_rate=0.1, classifier__max_depth=4, classifier__min_child_weight=1, classifier__n_estimators=100, classifier__subsample=0.8; total time=   0.8s
[CV] END classifier__colsample_bytree=0.8, classifier__gamma=0, classifier__learning_rate=0.1, classifier__max_depth=4, classifier__min_child_weight=3, classifier__n_estimators=100, classifier__subsample=0.8; total time=   0.8s
[CV] END classifier__colsample_bytree=0.8, classifier__gamma=0, classifier__learning_rate=0.1, classifier__max_depth=4, classifier__min_child_weight=1, classifier__n_estimators=100, class

## 5.4 Show The Best Result

In [32]:
print("\n=== HASIL TUNING ===")
print("Kombinasi Parameter Terbaik:\n", grid_search.best_params_)
print("Skor Validasi Terbaik (F1-Macro):", grid_search.best_score_)


=== HASIL TUNING ===
Kombinasi Parameter Terbaik:
 {'classifier__colsample_bytree': 1.0, 'classifier__gamma': 0, 'classifier__learning_rate': 0.2, 'classifier__max_depth': 7, 'classifier__min_child_weight': 1, 'classifier__n_estimators': 200, 'classifier__subsample': 0.8}
Skor Validasi Terbaik (F1-Macro): 0.9740238776778165


## 5.5 Take The Best Result

In [33]:
best_pipeline = grid_search.best_estimator_


## 5.6 Evaluate Using Validation Set

In [34]:
val_results, val_cm, val_pred = evaluate_model(best_pipeline, X_val, y_val, "VALIDATION")


================ VALIDATION ================
Confusion Matrix:
[[111  10]
 [  7 263]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9407    0.9174    0.9289       121
           1     0.9634    0.9741    0.9687       270

    accuracy                         0.9565       391
   macro avg     0.9520    0.9457    0.9488       391
weighted avg     0.9563    0.9565    0.9564       391

Ringkasan Metrik:
          accuracy: 0.9565
 balanced_accuracy: 0.9457
         precision: 0.9634
            recall: 0.9741
          f1_score: 0.9687
           roc_auc: 0.9826


The Validation Set results here are exactly identical to the pre-tuning (baseline) results.

## 5.7 Evaluate Using Test Set

In [35]:
test_results, test_cm, test_pred = evaluate_model(best_pipeline, X_test, y_test, "TEST")


================ TEST ================
Confusion Matrix:
[[112  10]
 [  5 265]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9573    0.9180    0.9372       122
           1     0.9636    0.9815    0.9725       270

    accuracy                         0.9617       392
   macro avg     0.9605    0.9498    0.9549       392
weighted avg     0.9617    0.9617    0.9615       392

Ringkasan Metrik:
          accuracy: 0.9617
 balanced_accuracy: 0.9498
         precision: 0.9636
            recall: 0.9815
          f1_score: 0.9725
           roc_auc: 0.9803


The Test Set results here are very close to the pre-tuning (baseline) results.

# 6. Conclusion

## Executive Summary

The **XGBClassifier** model performs strongly for geospatial quality-control sampling, reaching **~96–97% accuracy** on unseen test data. The key finding from this iteration is the **Baseline Dominance Pattern**: hyperparameter tuning improved the internal cross-validation score, but the **Baseline (Pre-Tuning Default)** pipeline still edged out the tuned pipeline on the held-out Test set for accuracy, precision, recall, and F1.

**Key Decision:** Deploy the **Pre-Tuning (Default)** pipeline to production. It reaches higher Test accuracy (**96.94%** vs **96.17%**) and higher Recall for Class 1 (**98.52%** vs **98.15%**) using half the tree count of the tuned model, at lower computational cost. The one metric where tuning helped was ROC AUC, which improved slightly on both Validation and Test — this is discussed in the notes below.

---

## 1. Hyperparameter Configurations

| Parameter | Baseline (Default) | Tuned (Best Grid Search) | Status / Change |
| --- | --- | --- | --- |
| `n_estimators` | **100** | **200** | Doubled (increased complexity) |
| `learning_rate` | **0.3** | **0.2** | Reduced step size |
| `max_depth` | **6** | **7** | Increased by 1 |
| `subsample` | **1.0** | **0.8** | Reduced (row subsampling added) |
| `colsample_bytree` | **1.0** | **1.0** | Unchanged |
| `min_child_weight` | **1** | **1** | Unchanged |
| `gamma` | **0** | **0** | Unchanged |

Source: baseline params from the `pipeline_et` definition (Section 4.3); tuned params from `grid_search.best_params_` (Section 5.4).

---

## 2. Comprehensive Metrics Comparison

| Metric | Baseline (Val) | Tuned (Val) | Baseline (Test) | Tuned (Test) | Test Impact |
| --- | --- | --- | --- | --- | --- |
| **Accuracy** | 0.9565 | 0.9565 | **0.9694** | 0.9617 | Decreased (‑0.77 pp) |
| **Precision (Class 1)** | 0.9634 | 0.9634 | **0.9708** | 0.9636 | Decreased (‑0.72 pp) |
| **Recall (Class 1)** | 0.9741 | 0.9741 | **0.9852** | 0.9815 | Decreased (‑0.37 pp) |
| **F1-Score (Class 1)** | 0.9687 | 0.9687 | **0.9779** | 0.9725 | Decreased (‑0.54 pp) |
| **ROC AUC** | 0.9805 | **0.9826** | 0.9794 | **0.9803** | Improved (+0.09 pp) |

Source: `evaluate_model()` outputs in Sections 4.6/4.7 (baseline) and 5.6/5.7 (tuned). Note that Validation-set predictions are identical between the baseline and tuned pipelines for every metric except ROC AUC — the two models only diverge on the Test set.

---

## 3. Hyperparameter Tuning Analysis

* **Cross-validation optimism vs. holdout reality:** GridSearchCV selected `n_estimators=200`, `learning_rate=0.2`, `max_depth=7`, `subsample=0.8` based on the highest internal CV score (**F1-Macro = 0.9740**). On the standalone Test set, however, this configuration under-performed the simpler baseline on accuracy, precision, recall, and F1.
* **Small but consistent recall gap:** The baseline model correctly recalled slightly more true positives on Test (**98.52%**) than the tuned model (**98.15%**) — a difference of 4 misclassified positives out of 270, roughly comparable between the two models rather than a dramatic drop.
* **Where tuning did help:** ROC AUC — a measure of how well the model ranks/separates the two classes across all thresholds — improved marginally with tuning on both Validation (0.9805 → 0.9826) and Test (0.9794 → 0.9803). This suggests the tuned model's probability estimates are slightly better calibrated, even though its default-threshold classification metrics are slightly worse.
* **Likely explanation:** Adding more trees (200 vs 100) and subsampling rows (0.8 vs 1.0) reduced variance in the CV folds used during search, but on this dataset that also nudged classification decisions at the boundary slightly away from the baseline's already-strong default configuration.

---

## 4. Technical Insights: Geospatial Feature Engineering

The model's strong accuracy (>96% across all iterations) is driven primarily by the geospatial feature engineering, combined with XGBoost's regularized tree architecture:

1. **Projection accuracy (UTM 48S / EPSG:32748):** Converting geographic coordinates from degrees to meters enabled precise physical distance calculations (`dist_c1`, `dist_c2`, `dist_c3`) relative to each polygon's centroid.
2. **High-signal topology features:** Spatial containment flags (`in_1`, `in_2`, `in_3`, `all_inside`) combined with directional movement vectors (`move_angle_12`, `move_angle_23`) gave the tree splits clean, near-linear separation between valid and invalid points.
3. **Generalization:** Test accuracy stayed close to — and for the baseline, above — Validation accuracy for both models (e.g. baseline: 96.94% test vs 95.65% validation), indicating the spatial features generalize well rather than overfitting to the training split.

---

## 5. Deployment Recommendations & Action Plan

> ### Primary Decision: Deploy the Pre-Tuning (Default) Model
>
> The **baseline XGBoost pipeline** (`n_estimators=100`, all other parameters at default) is the recommended asset for production. It reaches **96.94% Test accuracy** and **98.52% Class 1 recall** with half the tree count of the tuned variant — better classification performance at lower inference cost.

### Next Steps

* **Model serialization:** Export the baseline pipeline (`imputer` + `scaler` + `XGBClassifier(n_estimators=100)`) with `joblib` or `pickle` for integration into the automated QC pipeline.
* **Boundary case error analysis:** Manually inspect the misclassified Test instances (4 false negatives, 8 false positives out of 392). Focus on points sitting directly on or near polygon boundaries, where GPS precision or movement-angle features are ambiguous.
* **Future tuning framework:** If hyperparameter tuning is revisited in a future model update:
  * Optimize the grid search directly against **Class 1 Recall** rather than F1-Macro, since minimizing missed invalid points matters more than overall balance for this QC use case.
  * Use **Stratified Repeated K-Fold Cross-Validation** to reduce the gap between internal CV scores and standalone holdout performance seen in this round.
  * Since tuning improved ROC AUC but not threshold-based metrics, also evaluate whether adjusting the classification **decision threshold** on the tuned model (instead of only tuning hyperparameters) closes the recall/precision gap while keeping its better-calibrated probabilities.


In [36]:
# Save the best model after tuning
best_model_filename = 'best_xgboost_spatial_model_Final.pkl'
with open(best_model_filename, 'wb') as file:
    pickle.dump(best_pipeline, file)
print(f"\nModel XGBoost terbaik berhasil diekspor ke: {best_model_filename}")



Model XGBoost terbaik berhasil diekspor ke: best_xgboost_spatial_model_Final.pkl
